In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from visualisation import *
from plotly.subplots import make_subplots
from figures.generate_data import load_image_point_cloud
from diffusion_geometry.classes.main import DiffusionGeometry


Create some data

In [ ]:
data1 = np.random.uniform(-1, 1, (1200, 2))
plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
line_width = 1

x, y = data1[:,0], data1[:,1]
f = 3*dg1.function(np.exp(-((x+0.2)**2 + (y+0.2)**2)/0.15))
f.coeffs[20:] = 0
f += dg1.function(np.sin(2*y + 3*x - 0))
# f.coeffs[20:] = 0
f11 = plot_scatter_2d(data1, color = f.to_ambient(), size=4)
f11.show()

f12 = plot_quiver_2d(data1, f.grad().to_ambient(), scale=0.02, line_width=line_width)
f12.show()

f13 = plot_scatter_2d(data1, color = f.laplacian().to_ambient(), size=4)
f13.show()

In [ ]:
data2 = load_image_point_cloud("../data/data8.jpeg", n=1200, threshold=0.9, intensity_weighted=True)

plot_scatter_2d(data2, color = 'black').show()
dg2 = DiffusionGeometry.from_point_cloud(data2)

In [ ]:
x, y = data2[:,0], data2[:,1]
f = dg2.function(np.exp(-((x-0.4)**2 + (y+0.2)**2)/0.15))
f.coeffs[20:] = 0

f21 = plot_scatter_2d(data2, color = f.to_ambient(), size=4)
f21.show()

f22 = plot_quiver_2d(data2, f.grad().to_ambient(), scale=0.03, line_width=line_width)
f22.show()

f23 = plot_scatter_2d(data2, color = f.laplacian().to_ambient(), size=4)
f23.show()



In [ ]:
data3 = np.random.uniform(-1, 1, (1400, 2)) * [1, 1.2]
plot_scatter_2d(data3, color = 'black').show()
dg3 = DiffusionGeometry.from_point_cloud(data3)

In [ ]:
x, y = data3.T
vortex1 = np.stack((-y-0.8, x-0.3), axis=1)
vortex1 /= np.linalg.norm(vortex1, axis=1, keepdims=True)**2 * 1

X = data3 - np.array([[-0.2, 0.7]])
X /= np.linalg.norm(X, axis=1, keepdims=True)**1.7 * 0.5

X = dg3.form(X, 1)

a = dg3.form(vortex1, 1) - 1*X
# a.coeffs[50:] = 0

f31 = plot_quiver_2d(data3, a.to_ambient(), scale=0.02, line_width=1)
f31.show()

f32 = plot_scatter_2d(data3, a.codifferential().to_ambient(), size=4)
f32.show()

scale = 0.0007

f33 = plot_quiver_2d(data3, a.down_laplacian().to_ambient(), scale=scale, line_width=line_width)
f33.show()

f41 = plot_2form_2d(data3, a.d().to_ambient())
f41.show()

f42 = plot_quiver_2d(data3, a.up_laplacian().to_ambient(), scale=scale, line_width=line_width)
f42.show()

f43 = plot_quiver_2d(data3, a.laplacian().to_ambient(), scale=scale, line_width=line_width)
f43.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

tall = 1.2
total_height = 2 + 2 * tall

fig = make_subplots(
    rows=4,
    cols=3,
    horizontal_spacing=0.02,
    vertical_spacing=0.03,
    row_heights=[1 / total_height, 1 / total_height, tall / total_height, tall / total_height],
    shared_xaxes=True,
    # shared_yaxes=True,
)

# --- Add traces ---
fig.add_traces(list(f11.data), rows=[1]*len(f11.data), cols=[1]*len(f11.data))
fig.add_traces(list(f12.data), rows=[1]*len(f12.data), cols=[2]*len(f12.data))
fig.add_traces(list(f13.data), rows=[1]*len(f13.data), cols=[3]*len(f13.data))

fig.add_traces(list(f21.data), rows=[2]*len(f21.data), cols=[1]*len(f21.data))
fig.add_traces(list(f22.data), rows=[2]*len(f22.data), cols=[2]*len(f22.data))
fig.add_traces(list(f23.data), rows=[2]*len(f23.data), cols=[3]*len(f23.data))

fig.add_traces(list(f31.data), rows=[3]*len(f31.data), cols=[1]*len(f31.data))
fig.add_traces(list(f32.data), rows=[3]*len(f32.data), cols=[2]*len(f32.data))
fig.add_traces(list(f33.data), rows=[3]*len(f33.data), cols=[3]*len(f33.data))

fig.add_traces(list(f41.data), rows=[4]*len(f41.data), cols=[1]*len(f41.data))
fig.add_traces(list(f42.data), rows=[4]*len(f42.data), cols=[2]*len(f42.data))
fig.add_traces(list(f43.data), rows=[4]*len(f43.data), cols=[3]*len(f43.data))

# --- Common axis ranges ---
fig.update_xaxes(range=[-1, 1], row=1)
fig.update_yaxes(range=[-1, 1], row=1)
fig.update_xaxes(range=[-1, 1], row=2)
fig.update_yaxes(range=[-1, 1], row=2)
fig.update_xaxes(range=[-1, 1], row=3)
fig.update_yaxes(range=[-tall, tall], row=3)
fig.update_xaxes(range=[-1, 1], row=4)
fig.update_yaxes(range=[-tall, tall], row=4)

# --- Layout ---
fig.update_layout(
    width=1000,
    height= int(1000 * total_height / 3) + 100,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)
fig.update_xaxes(scaleanchor="y", scaleratio=1)
clean_fig(fig)

for axis in fig.layout:
    if axis.startswith("xaxis") or axis.startswith("yaxis"):
        fig.layout[axis].automargin = False

fig.show()
fig.write_image("figs/summary_2x4.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1:
        if col == 1:
            return rf"$f$"
        elif col == 2:
            return rf"$d^{{(0)}} f$"
        elif col == 3:
            return rf"$\Delta^{{(0)}} f = \partial^{{(1)}} d^{{(0)}} f$"
    if row == 2:
        if col == 1:
            return rf"$h$"
        elif col == 2:
            return rf"$d^{{(0)}} h$"
        elif col == 3:
            return rf"$\Delta^{{(0)}} h = \partial^{{(1)}} d^{{(0)}} h$"
    if row == 3:
        if col == 1:
            return rf"$\alpha$"
        if col == 2:
            return rf"$\partial^{{(1)}} \alpha$"
        elif col == 3:
            return rf"$d^{{(0)}} \partial^{{(1)}} \alpha$"
    if row == 4:
        if col == 1:
            return rf"$d^{{(1)}} \alpha$"
        elif col == 2:
            return rf"$\partial^{{(2)}} d^{{(1)}} \alpha$"
        elif col == 3:
            return rf"$\Delta^{{(1)}} \alpha = (d^{{(0)}} \partial^{{(1)}} + \partial^{{(2)}} d^{{(1)}}) \alpha$"
    else:
        return ""

print('top two rows')
overpic_labels(fig, labels, 
               stretch_x=1.0, 
            #    offset_x=-2,
               stretch_y=0.92,
               offset_y=9.5)

print('\n\nbottom two rows')
overpic_labels(fig, labels, 
               stretch_x=1., 
            #    offset_x=-2,
               stretch_y=1.,
               offset_y=13.8)


Hodge decomp

In [ ]:
data4 = load_image_point_cloud("../data/data9.jpeg", n=700, threshold=0.9, intensity_weighted=True) * [1, 1.2]
plot_scatter_2d(data4, color = 'black').show()
dg4 = DiffusionGeometry.from_point_cloud(data4)

In [ ]:
f = dg4.function_space.zeros()
f.coeffs[3] = 1
# f.coeffs[1] = 2
# f.coeffs[2] = 1
f = dg4.function(data4[:,0] + data4[:,1])

coexact, harmonic = f.hodge_decomposition()

amax = np.max(np.abs(f.to_ambient()))
f11 = plot_scatter_2d(data4, f.to_ambient(), size=4, amax=amax)
f11.show()
f12 = plot_scatter_2d(data4, harmonic.to_ambient(), size=4, amax=amax)
f12.show()
f13 = plot_scatter_2d(data4, coexact.codifferential().to_ambient(), size=4, amax=amax)
f13.show()

In [ ]:
data5 = np.random.uniform(-1, 1, (1400, 2)) * [1, 1]
plot_scatter_2d(data5, color = 'black').show()
dg5 = DiffusionGeometry.from_point_cloud(data5)

In [ ]:
x, y = data5.T
vortex1 = np.stack((-y-0.8, x-0.3), axis=1)
vortex1 /= np.linalg.norm(vortex1, axis=1, keepdims=True)**2 * 1

vortex2 = np.stack((y-0.8, -x-0.3), axis=1)
vortex2 /= np.linalg.norm(vortex2, axis=1, keepdims=True)**2 * 1

X = data5 - np.array([[-0.1, 0.7]])
X /= np.linalg.norm(X, axis=1, keepdims=True)**1.7 * 0.5

X = dg5.form(X, 1)

a = dg5.form(vortex1, 1) + dg5.form(vortex2, 1) - 1*X

exact, coexact, harmonic = a.hodge_decomposition()

scale = 0.02

f21 = plot_quiver_2d(data5, a.to_ambient(), scale=scale, line_width=1)
f21.show()

f22 = plot_quiver_2d(data5, exact.d().to_ambient(), scale=scale, line_width=1)
f22.show()

f23 = plot_quiver_2d(data5, coexact.codifferential().to_ambient(), scale=scale, line_width=1)
f23.show()

# f41 = plot_2form_2d(data5, a.d().to_ambient())
# f41.show()

# f42 = plot_quiver_2d(data5, a.up_laplacian().to_ambient(), scale=scale, line_width=line_width)
# f42.show()

# f43 = plot_quiver_2d(data5, a.laplacian().to_ambient(), scale=scale, line_width=line_width)
# f43.show()


In [ ]:
from generate_data import gen_3d_data

n = 2000
data6 = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = n, noise = 0.0, seed = 0)[0]
# data6 = data6[:, [1, 2, 0]]
dg6 = DiffusionGeometry.from_point_cloud(data6)
camera = dict(eye=dict(x=0.8, y=-0.9, z=1.),
                center=dict(x=0, y=0, z=-0.2),
                up=dict(x=0, y=0, z=1))

plot_scatter_3d(data6, color='black', camera=camera).show()

In [ ]:
evals, evecs = dg6.laplacian(1).spectrum()
a = -evecs[1]
f = dg6.function(data6[:,1]) * 1
a = f.d() - a

scale = 0.1
arrow_scale = 0.6
f31 = plot_quiver_3d(data6, a.to_ambient(), scale=scale, line_width=1, camera=camera, arrow_scale=arrow_scale)
f31.show()
exact, coexact, harmonic = a.hodge_decomposition()

print(exact.norm())
f32 = plot_quiver_3d(data6, exact.d().to_ambient(), scale=scale, line_width=1, camera=camera, arrow_scale=0.6*arrow_scale)
f32.show()

print(coexact.norm())
f33 = plot_quiver_3d(data6, coexact.codifferential().to_ambient(), scale=scale, line_width=1, camera=camera, arrow_scale=1*arrow_scale)
f33.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=3,
    cols=3,
    specs=specs,
    horizontal_spacing=0.02,
    vertical_spacing=0.03,
    row_heights=[0.25, 0.4, 0.25],
    shared_yaxes=True
)

# --- Add traces row by row ---
fig.add_traces(list(f11.data), rows=[1] * len(f11.data), cols=[1] * len(f11.data))
fig.add_traces(list(f12.data), rows=[1] * len(f12.data), cols=[2] * len(f12.data))
fig.add_traces(list(f13.data), rows=[1] * len(f13.data), cols=[3] * len(f13.data))

fig.add_traces(list(f21.data), rows=[2] * len(f21.data), cols=[1] * len(f21.data))
fig.add_traces(list(f22.data), rows=[2] * len(f22.data), cols=[2] * len(f22.data))
fig.add_traces(list(f23.data), rows=[2] * len(f23.data), cols=[3] * len(f23.data))

fig.add_traces(list(f31.data), rows=[3] * len(f31.data), cols=[1] * len(f31.data))
fig.add_traces(list(f32.data), rows=[3] * len(f32.data), cols=[2] * len(f32.data))
fig.add_traces(list(f33.data), rows=[3] * len(f33.data), cols=[3] * len(f33.data))

# --- Synchronise 2D axis ranges ---
xrange = [min(data4[:,0]), max(data4[:,0])]
yrange = [min(data4[:,1]), max(data4[:,1])]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)

middle_row = [f21, f22, f23]
x_vals, y_vals = [], []
for f in middle_row:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Synchronise 3D ranges ---
all_3d = [f31, f32, f33]
x3d, y3d, z3d = [], [], []
for f in all_3d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 4):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1000,
    height=1000,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)
fig.update_xaxes(scaleanchor="y", scaleratio=1)
clean_fig(fig)
fig.update_xaxes(constrain="domain")
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1:
        if col == 1:
            return rf"$f$"
        elif col == 2:
            return rf"harmonic part of $f$"
        elif col == 3:
            return rf"coexact part of $f$"
    if row == 2:
        if col == 1:
            return rf"$\alpha$"
        elif col == 2:
            return rf"exact part of $\alpha$"
        elif col == 3:
            return rf"coexact part of $\alpha$"
    if row == 3:
        if col == 1:
            return rf"$\beta$"
        elif col == 2:
            return rf"exact part of $\beta$"
        elif col == 3:
            return rf"coexact part of $\beta$"
    else:
        return ""
    
overpic_labels(fig, labels, 
               stretch_x=1.0, 
               stretch_y=1.01,
               offset_y=13.5)